# Day 42 — Randomized algorithms and Monte Carlo: estimating with dice, and testing code that rolls them

Throw a million random darts at a square and count how many land inside the quarter
circle: four times that fraction is pi to about three decimals. The error shrinks like
`1 / sqrt(N)`, so every extra correct digit costs a hundred times more darts.

Randomized code has one nasty property: when it is wrong it does not crash. It still
returns a random-looking answer, just drawn from the wrong distribution. This notebook
builds a tool that tests randomness *exactly* - by replaying every possible sequence of
random choices - and uses it on shuffling (LeetCode 384), reservoir sampling (382, 398),
rejection sampling (470) and weighted picking (528).

In [1]:
from __future__ import annotations

import math
import random
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from fractions import Fraction
from itertools import accumulate, permutations

## 1. Monte Carlo: estimating pi

A dart lands inside the quarter circle with probability pi/4. One dart's estimate
`4 * [inside]` has standard deviation `4 * sqrt(p(1-p)) = 1.642`, so the average of N
darts is off by about `1.642 / sqrt(N)`.

In [2]:
# ------------------------------------------------------------ 1. Monte Carlo pi
def estimate_pi(n: int, rng: random.Random) -> float:
    """Throw n darts at the unit square; the quarter circle covers pi/4 of it."""
    inside = 0
    for _ in range(n):
        x, y = rng.random(), rng.random()
        if x * x + y * y < 1.0:
            inside += 1
    return 4 * inside / n


P_IN = math.pi / 4


SIGMA_PI = 4 * math.sqrt(P_IN * (1 - P_IN))       # std of ONE dart's 4*[inside]


def pi_std(n: int) -> float:
    """Standard error of estimate_pi(n): 1.642 / sqrt(n)."""
    return SIGMA_PI / math.sqrt(n)


def darts_for_error(err: float) -> float:
    """Darts needed so that one standard error equals err."""
    return (SIGMA_PI / err) ** 2


rng = random.Random(42)
for n in (100, 10_000, 1_000_000):
    est = estimate_pi(n, rng)
    print(f'{n:>9,} darts: pi ~ {est:.6f}   error {abs(est - math.pi):.6f}   one standard error {pi_std(n):.6f}')
for n, trials in ((100, 400), (1_000, 200), (10_000, 100), (100_000, 20)):
    errs = [estimate_pi(n, rng) - math.pi for _ in range(trials)]
    print(f'N = {n:>7,}: rms error {math.sqrt(sum(e * e for e in errs) / trials):.5f}   law {pi_std(n):.5f}')
print(f'six correct decimals needs {darts_for_error(5e-7):.2e} darts')

      100 darts: pi ~ 3.120000   error 0.021593   one standard error 0.164218
   10,000 darts: pi ~ 3.125200   error 0.016393   one standard error 0.016422
1,000,000 darts: pi ~ 3.140828   error 0.000765   one standard error 0.001642
N =     100: rms error 0.16296   law 0.16422
N =   1,000: rms error 0.05454   law 0.05193
N =  10,000: rms error 0.01690   law 0.01642
N = 100,000: rms error 0.00389   law 0.00519
six correct decimals needs 1.08e+13 darts


## 2. Las Vegas vs Monte Carlo

*Las Vegas*: always correct, random running time - quicksort with a random pivot.
*Monte Carlo*: fixed running time, small chance of a wrong answer - Miller-Rabin.
Fermat's test (Day 41's `pow`) is fooled by Carmichael numbers like 561 for most bases;
Miller-Rabin also checks the square roots of 1 along the way, which caps the liars at 1/4.

In [3]:
# --------------------------------------------- 2a. Las Vegas: random pivots
def quicksort_comparisons(a: list, rng: random.Random | None) -> tuple[list, int]:
    """Lomuto quicksort with an explicit stack.  rng=None -> first element as
    pivot; otherwise a random pivot.  Returns (sorted copy, comparisons)."""
    a = list(a)
    comps = 0
    stack = [(0, len(a) - 1)]
    while stack:
        lo, hi = stack.pop()
        if lo >= hi:
            continue
        if rng is not None:
            k = rng.randint(lo, hi)
            a[lo], a[k] = a[k], a[lo]
        pivot, i = a[lo], lo
        for j in range(lo + 1, hi + 1):
            comps += 1
            if a[j] < pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
        a[lo], a[i] = a[i], a[lo]
        stack.append((lo, i - 1))
        stack.append((i + 1, hi))
    return a, comps


def expected_quicksort_comparisons(n: int) -> float:
    """Exact mean for a uniformly random pivot: 2(n+1)H_n - 4n."""
    h = sum(1 / k for k in range(1, n + 1))
    return 2 * (n + 1) * h - 4 * n


n = 2000
_, first = quicksort_comparisons(list(range(n)), None)
runs = [quicksort_comparisons(list(range(n)), random.Random(s))[1] for s in range(200)]
print(f'sorted input, first pivot: {first:,} comparisons')
print(f'random pivot, 200 runs: mean {sum(runs)/len(runs):,.0f}  min {min(runs):,}  max {max(runs):,}'
      f'   exact mean {expected_quicksort_comparisons(n):,.0f}')

sorted input, first pivot: 1,999,000 comparisons
random pivot, 200 runs: mean 24,795  min 22,449  max 31,811   exact mean 24,730


In [4]:
# ------------------------------------------- 2b. Monte Carlo: primality tests
def fermat_passes(n: int, a: int) -> bool:
    """Fermat: if n is prime then a^(n-1) = 1 (mod n).  The converse is false."""
    return pow(a, n - 1, n) == 1


def miller_rabin_passes(n: int, a: int) -> bool:
    """One Miller-Rabin round with base a.  False means n is certainly composite."""
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    x = pow(a, d, n)
    if x == 1 or x == n - 1:
        return True
    for _ in range(s - 1):
        x = x * x % n
        if x == n - 1:
            return True
    return False                 # a square root of 1 other than +-1: composite


def is_probable_prime(n: int, rounds: int, rng: random.Random) -> bool:
    """Monte Carlo: always says True for a prime; wrong on a composite with
    probability at most (1/4)^rounds."""
    if n < 4:
        return n in (2, 3)
    if n % 2 == 0:
        return False
    return all(miller_rabin_passes(n, rng.randrange(2, n - 1)) for _ in range(rounds))


SMALL_PRIMES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)


def is_prime(n: int) -> bool:
    """Deterministic for n < 3.18 * 10**23 (every 64-bit integer): with the twelve
    primes 2..37 as bases no composite below that bound passes."""
    if n < 2:
        return False
    for p in SMALL_PRIMES:
        if n % p == 0:
            return n == p
    return all(miller_rabin_passes(n, a) for a in SMALL_PRIMES)


def sieve(n: int) -> bytearray:
    s = bytearray([1]) * n
    s[:2] = b'\x00\x00'
    for p in range(2, math.isqrt(n - 1) + 1):
        if s[p]:
            s[p * p::p] = bytes(len(range(p * p, n, p)))
    return s


def liars(n: int) -> tuple[int, int, int]:
    """(bases tried, Fermat liars, Miller-Rabin liars) over bases 2..n-2."""
    bases = range(2, n - 1)
    return (len(bases), sum(fermat_passes(n, a) for a in bases),
            sum(miller_rabin_passes(n, a) for a in bases))


tried, f_liars, mr_liars = liars(561)
print(f'561: Fermat fooled by {f_liars}/{tried} bases, Miller-Rabin by {mr_liars}/{tried}')
s = sieve(100_000)
rng = random.Random(1)
print('wrong answers below 100,000 with 1 / 5 random bases:',
      sum(is_probable_prime(m, 1, rng) != bool(s[m]) for m in range(100_000)),
      sum(is_probable_prime(m, 5, rng) != bool(s[m]) for m in range(100_000)))
print('is_prime(2**61 - 1) =', is_prime(2**61 - 1))

561: Fermat fooled by 318/558 bases, Miller-Rabin by 8/558
wrong answers below 100,000 with 1 / 5 random bases: 38 0
is_prime(2**61 - 1) = True


## 3. Testing randomness exactly

Replace the random source with a function `rr(k)` that the tester controls. Run the
algorithm once for every possible sequence of answers (an odometer over the choices) and
add up each outcome's probability as an exact `Fraction`. A 1/27 bias cannot hide in noise.

In [5]:
# ------------------------------------------ 3. exact distribution of a sampler
def exact_distribution(run) -> dict:
    """run(randrange) must call randrange(k) for its random choices and return
    a hashable result.  Every possible sequence of choices is replayed once
    (an odometer over the choices) and the probabilities are added up as exact
    fractions - no sampling, no noise, so a bias of 1/27 cannot hide."""
    dist: dict = defaultdict(Fraction)
    prefix: list[int] = []
    while True:
        radices: list[int] = []

        def randrange(k: int) -> int:
            pos = len(radices)
            radices.append(k)
            if pos == len(prefix):
                prefix.append(0)
            return prefix[pos]

        out = run(randrange)
        p = Fraction(1)
        for k in radices:
            p /= k
        dist[out] += p
        del prefix[len(radices):]
        while prefix and prefix[-1] + 1 == radices[len(prefix) - 1]:
            prefix.pop()
        if not prefix:
            return dict(dist)
        prefix[-1] += 1


def frac_str(f: Fraction) -> str:
    return '0' if f == 0 else f'{f.numerator}/{f.denominator}'


coin = exact_distribution(lambda rr: rr(2) + rr(2))
print('two coin flips, heads counted:', {k: frac_str(v) for k, v in sorted(coin.items())})

two coin flips, heads counted: {0: '1/4', 1: '1/2', 2: '1/4'}


## 4. Shuffling - LeetCode 384

Swapping every slot with *any* slot makes `n^n` equally likely runs; 27 runs cannot split
evenly over 6 orders. Writing `randrange(i)` instead of `randrange(i + 1)` turns
Fisher-Yates into Sattolo's algorithm: no item ever stays put.

In [6]:
# --------------------------------------------------------------- 4. shuffling
def shuffle_naive(a: list, randrange) -> list:
    """Swap every slot with a random slot ANYWHERE: n**n equally likely runs."""
    a = list(a)
    n = len(a)
    for i in range(n):
        j = randrange(n)
        a[i], a[j] = a[j], a[i]
    return a


def shuffle_off_by_one(a: list, randrange) -> list:
    """Fisher-Yates with randrange(i) instead of randrange(i + 1): j < i, so the
    element at i can never stay - this is Sattolo's algorithm (one big cycle)."""
    a = list(a)
    for i in range(len(a) - 1, 0, -1):
        j = randrange(i)
        a[i], a[j] = a[j], a[i]
    return a


def fisher_yates(a: list, randrange) -> list:
    """Fill slots from the back; slot i gets a uniform pick of the i+1 still free."""
    a = list(a)
    for i in range(len(a) - 1, 0, -1):
        j = randrange(i + 1)           # 0..i inclusive - i itself is allowed
        a[i], a[j] = a[j], a[i]
    return a


def shuffle_report(fn, n: int) -> tuple[Counter, int]:
    """How many of the equally likely random runs produce each order."""
    dist = exact_distribution(lambda rr: tuple(fn(range(n), rr)))
    runs = math.lcm(*(f.denominator for f in dist.values()))
    return Counter({p: int(f * runs) for p, f in dist.items()}), runs


class Solution384:
    """LeetCode 384 - Shuffle an Array."""

    def __init__(self, nums: list[int]):
        self.original = list(nums)
        self.rng = random.Random()

    def reset(self) -> list[int]:
        return list(self.original)

    def shuffle(self) -> list[int]:
        a = list(self.original)
        for i in range(len(a) - 1, 0, -1):
            j = self.rng.randrange(i + 1)
            a[i], a[j] = a[j], a[i]
        return a


for fn in (shuffle_naive, shuffle_off_by_one, fisher_yates):
    cnt, runs = shuffle_report(fn, 3)
    print(f'{fn.__name__:<20} {runs:>2} runs ->',
          '  '.join(f"{''.join('abc'[i] for i in p)}:{cnt.get(p, 0)}" for p in permutations(range(3))))
s = Solution384([1, 2, 3]); s.rng.seed(3)
print('LC 384:', s.shuffle(), s.reset(), s.shuffle())

shuffle_naive        27 runs -> abc:4  acb:5  bac:5  bca:5  cab:4  cba:4
shuffle_off_by_one    2 runs -> abc:0  acb:0  bac:0  bca:1  cab:1  cba:0
fisher_yates          6 runs -> abc:1  acb:1  bac:1  bca:1  cab:1  cba:1
LC 384: [2, 3, 1] [1, 2, 3] [1, 3, 2]


## 5. Reservoir sampling - LeetCode 382 and 398

Keep the i-th item with chance 1/i. Item j survives with
`1/j * j/(j+1) * ... * (n-1)/n = 1/n`. With 0-based `i` the second item is kept with
chance 1/1, so the first item never comes back.

In [7]:
# ------------------------------------------------------ 5. reservoir sampling
def reservoir_one(stream, randrange):
    """Keep the i-th item (1-based) with probability 1/i.  One pass, O(1) memory,
    and the length of the stream is never needed."""
    chosen = None
    for i, x in enumerate(stream, 1):
        if randrange(i) == 0:          # 1 chance in i
            chosen = x
    return chosen


def reservoir_off_by_one(stream, randrange):
    """Same loop with a 0-based i: item i is kept with probability 1/i, so the
    first item (i = 0) is skipped and the second (i = 1) ALWAYS replaces it."""
    chosen = None
    for i, x in enumerate(stream):
        if i == 0:
            chosen = x
            continue
        if randrange(i) == 0:          # should be randrange(i + 1)
            chosen = x
    return chosen


def reservoir_k(stream, k: int, randrange) -> list:
    """Algorithm R: keep k items; item i (0-based, i >= k) enters with chance k/(i+1)."""
    res = []
    for i, x in enumerate(stream):
        if i < k:
            res.append(x)
        else:
            j = randrange(i + 1)
            if j < k:
                res[j] = x
    return res


class ListNode:
    def __init__(self, val: int = 0, next: 'ListNode | None' = None):
        self.val, self.next = val, next


def build_list(vals) -> ListNode | None:
    head = None
    for v in reversed(vals):
        head = ListNode(v, head)
    return head


class Solution382:
    """LeetCode 382 - Linked List Random Node (reservoir, no length needed)."""

    def __init__(self, head: ListNode | None):
        self.head = head
        self.rng = random.Random()

    def getRandom(self) -> int:
        chosen, i, node = 0, 0, self.head
        while node:
            i += 1
            if self.rng.randrange(i) == 0:
                chosen = node.val
            node = node.next
        return chosen


class Solution398:
    """LeetCode 398 - Random Pick Index: reservoir over the matching positions."""

    def __init__(self, nums: list[int]):
        self.nums = nums
        self.rng = random.Random()

    def pick(self, target: int) -> int:
        chosen, seen = -1, 0
        for i, v in enumerate(self.nums):
            if v == target:
                seen += 1
                if self.rng.randrange(seen) == 0:
                    chosen = i
        return chosen


items = list('abcde')
for fn in (reservoir_one, reservoir_off_by_one):
    d = exact_distribution(lambda rr: fn(items, rr))
    print(f'{fn.__name__:<22}', {x: frac_str(d.get(x, Fraction(0))) for x in items})
s = Solution382(build_list([1, 2, 3])); s.rng.seed(5)
print('LC 382:', dict(sorted(Counter(s.getRandom() for _ in range(30_000)).items())))
s = Solution398([1, 2, 3, 3, 3]); s.rng.seed(9)
print('LC 398 pick(3):', dict(sorted(Counter(s.pick(3) for _ in range(30_000)).items())), ' pick(1):', s.pick(1))

reservoir_one          {'a': '1/5', 'b': '1/5', 'c': '1/5', 'd': '1/5', 'e': '1/5'}
reservoir_off_by_one   {'a': '0', 'b': '1/4', 'c': '1/4', 'd': '1/4', 'e': '1/4'}
LC 382: {1: 9977, 2: 9958, 3: 10065}
LC 398 pick(3): {2: 9919, 3: 10010, 4: 10071}  pick(1): 0


## 6. Rejection sampling - LeetCode 470

Two `rand7` calls give 49 equally likely cells. Keep the first 40 (four copies of 1..10),
throw the other 9 away and draw again: `2 x 49/40 = 2.45` calls on average.

In [8]:
# ------------------------------------------------------ 6. rejection sampling
def rand10_from_rand7(rand7) -> tuple[int, int]:
    """Two rand7 calls give 49 equally likely cells; keep the first 40 (four
    copies of 1..10) and throw the other 9 away.  Returns (digit, calls)."""
    calls = 0
    while True:
        idx = (rand7() - 1) * 7 + rand7()       # 1..49, uniform
        calls += 2
        if idx <= 40:
            return (idx - 1) % 10 + 1, calls


def rand10_mod_bias(rand7) -> int:
    """No rejection: 49 is not a multiple of 10, so one digit is short a cell."""
    idx = (rand7() - 1) * 7 + rand7()
    return idx % 10 + 1


def rand10_reuse(rand7) -> tuple[int, int]:
    """Keep the leftover randomness of a rejected draw instead of discarding it."""
    calls = 0
    while True:
        a = (rand7() - 1) * 7 + rand7()          # 1..49
        calls += 2
        if a <= 40:
            return (a - 1) % 10 + 1, calls
        a = (a - 40 - 1) * 7 + rand7()           # 1..63
        calls += 1
        if a <= 60:
            return (a - 1) % 10 + 1, calls
        a = (a - 60 - 1) * 7 + rand7()           # 1..21
        calls += 1
        if a <= 20:
            return (a - 1) % 10 + 1, calls


def rand10_one_round(randrange) -> int | None:
    """One round of the rejection sampler, for exact_distribution (None = reject)."""
    idx = randrange(7) * 7 + randrange(7) + 1
    return (idx - 1) % 10 + 1 if idx <= 40 else None


class Solution470:
    """LeetCode 470 - Implement Rand10() Using Rand7()."""

    def __init__(self, rand7):
        self.rand7 = rand7

    def rand10(self) -> int:
        while True:
            idx = (self.rand7() - 1) * 7 + self.rand7()
            if idx <= 40:
                return (idx - 1) % 10 + 1


one = exact_distribution(rand10_one_round)
print('one round:', {k: frac_str(v) for k, v in one.items()})
bias = exact_distribution(lambda rr: rand10_mod_bias(lambda: rr(7) + 1))
print('no rejection:', {k: frac_str(v) for k, v in sorted(bias.items())})
sim = random.Random(11); r7 = lambda: sim.randint(1, 7)
c1 = [rand10_from_rand7(r7)[1] for _ in range(200_000)]
c2 = [rand10_reuse(r7)[1] for _ in range(200_000)]
print(f'average calls over 200,000: {sum(c1)/len(c1):.3f}   reusing leftovers: {sum(c2)/len(c2):.3f}')

one round: {1: '4/49', 2: '4/49', 3: '4/49', 4: '4/49', 5: '4/49', 6: '4/49', 7: '4/49', 8: '4/49', 9: '4/49', 10: '4/49', None: '9/49'}
no rejection: {1: '4/49', 2: '5/49', 3: '5/49', 4: '5/49', 5: '5/49', 6: '5/49', 7: '5/49', 8: '5/49', 9: '5/49', 10: '5/49'}
average calls over 200,000: 2.456   reusing leftovers: 2.193


## 7. Weighted pick - LeetCode 528

Prefix sums cut `0..total-1` into slices, one per index, as wide as its weight;
`bisect_right(prefix, randrange(total))` finds the slice. An LLM samples its next token
from `softmax(logits / temperature)` with exactly this step.

In [9]:
# ---------------------------------------------- 7. weighted pick, prefix sums
class Solution528:
    """LeetCode 528 - Random Pick with Weight.  O(n) build, O(log n) per pick."""

    def __init__(self, w: list[int], rng: random.Random | None = None):
        self.prefix = list(accumulate(w))        # w=[1,3] -> [1, 4]
        self.rng = rng or random.Random()

    def pickIndex(self) -> int:
        target = self.rng.randrange(self.prefix[-1])   # 0..total-1
        return bisect_right(self.prefix, target)       # first prefix > target


def pick_index_exact(w, randrange) -> int:
    prefix = list(accumulate(w))
    return bisect_right(prefix, randrange(prefix[-1]))


def pick_index_off_by_one(w, randrange) -> int:
    """randint(0, total) is total + 1 values - one too many, and the extra one
    (target 0) lands on index 0."""
    prefix = list(accumulate(w))
    return bisect_left(prefix, randrange(prefix[-1] + 1))


for w in ([1, 3], [1, 99]):
    good = exact_distribution(lambda rr: pick_index_exact(w, rr))
    bad = exact_distribution(lambda rr: pick_index_off_by_one(w, rr))
    print(w, 'correct', [frac_str(good[i]) for i in range(len(w))],
          ' randint + bisect_left', [frac_str(bad[i]) for i in range(len(w))])
s = Solution528([1, 3], random.Random(13))
print('LC 528:', dict(sorted(Counter(s.pickIndex() for _ in range(40_000)).items())))
for temp in (0.5, 1.0, 2.0):
    z = [math.exp(v / temp) for v in (2.0, 1.0, 0.1)]
    print(f'temperature {temp}:', [round(x / sum(z), 3) for x in z])

[1, 3] correct ['1/4', '3/4']  randint + bisect_left ['2/5', '3/5']
[1, 99] correct ['1/100', '99/100']  randint + bisect_left ['2/101', '99/101']
LC 528: {0: 10011, 1: 29989}
temperature 0.5: [0.864, 0.117, 0.019]
temperature 1.0: [0.659, 0.242, 0.099]
temperature 2.0: [0.502, 0.304, 0.194]


## Tests

In [10]:
# ====================================================================== tests
def run_tests() -> None:
    rng = random.Random(0)
    for n in range(1, 7):
        cnt, runs = shuffle_report(fisher_yates, n)
        assert len(cnt) == math.factorial(n) and set(cnt.values()) == {1}
        cnt, _ = shuffle_report(shuffle_off_by_one, n)
        assert len(cnt) == math.factorial(n - 1)
        assert all(all(p[i] != i for i in range(n)) for p in cnt) or n == 1
    cnt, runs = shuffle_report(shuffle_naive, 3)
    assert runs == 27 and sorted(cnt.values()) == [4, 4, 4, 5, 5, 5]
    for n in range(1, 7):
        d = exact_distribution(lambda rr: reservoir_one(range(n), rr))
        assert all(v == Fraction(1, n) for v in d.values()) and len(d) == n
    d = exact_distribution(lambda rr: reservoir_off_by_one(range(5), rr))
    assert 0 not in d and all(v == Fraction(1, 4) for v in d.values())
    for n in range(2, 7):
        for k in range(1, n + 1):
            d = exact_distribution(lambda rr: frozenset(reservoir_k(range(n), k, rr)))
            assert len(d) == math.comb(n, k) and set(d.values()) == {Fraction(1, math.comb(n, k))}
    one = exact_distribution(rand10_one_round)
    assert one[None] == Fraction(9, 49) and all(one[k] == Fraction(4, 49) for k in range(1, 11))
    bias = exact_distribution(lambda rr: rand10_mod_bias(lambda: rr(7) + 1))
    assert bias[1] == Fraction(4, 49) and all(bias[k] == Fraction(5, 49) for k in range(2, 11))
    for w in ([1], [1, 3], [2, 5, 3], [1, 99], [4, 4, 4]):
        d = exact_distribution(lambda rr: pick_index_exact(w, rr))
        assert all(d.get(i, 0) == Fraction(w[i], sum(w)) for i in range(len(w)))
    d = exact_distribution(lambda rr: pick_index_off_by_one([1, 99], rr))
    assert d[0] == Fraction(2, 101)
    s = sieve(50_000)
    assert all(is_prime(m) == bool(s[m]) for m in range(50_000))
    assert is_prime(2**61 - 1) and not is_prime(2**61 + 1) and is_prime(2**89 - 1)
    assert all(fermat_passes(561, a) for a in range(2, 560) if math.gcd(a, 561) == 1)
    for m in range(9, 600, 2):
        if not s[m]:
            t, _, mr = liars(m)
            assert mr * 4 <= t + 2         # Rabin's bound, bases 2..n-2
    data = list(range(300))
    rng.shuffle(data)
    for r in (None, random.Random(1)):
        out, c = quicksort_comparisons(data, r)
        assert out == sorted(data)
    assert quicksort_comparisons(list(range(100)), None)[1] == 100 * 99 // 2
    s = Solution384([1, 2, 3, 4])
    assert sorted(s.shuffle()) == [1, 2, 3, 4] and s.reset() == [1, 2, 3, 4]
    s = Solution382(build_list([7]))
    assert s.getRandom() == 7
    s = Solution398([1, 2, 3, 3, 3])
    assert s.pick(1) == 0 and s.pick(3) in (2, 3, 4)
    s = Solution470(lambda: rng.randint(1, 7))
    assert all(1 <= s.rand10() <= 10 for _ in range(1000))
    s = Solution528([1, 3])
    assert all(s.pickIndex() in (0, 1) for _ in range(100))


run_tests()
print('all assertions passed')

all assertions passed
